In [1]:
!pip install --upgrade torchao
!pip uninstall -y transformers accelerate peft
!pip install transformers==5.13.1
!pip install accelerate==1.14.0
!pip install peft==0.19.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 38.5 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
Found existing installation: peft 0.19.1
Uninstalling peft-0.19.1:
  Successfully uninstalled peft-0.19.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 95.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 34.6 MB/s eta 0:00:00
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.7.0
    Uninstalling safetensors-0.7.0:
      Successfully uninstalled safetensors-0.7.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 7.2 MB/s eta 0:0

In [2]:
import torch
import transformers
import accelerate
import peft
import torchao

print(torch.__version__)
print(transformers.__version__)
print(accelerate.__version__)
print(peft.__version__)
print(torchao.__version__)

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


2.10.0+cu128
5.13.1
1.14.0
0.19.1
0.17.0


In [3]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('smart-mcq-solver-challenge')

print("Path to competition files:", path)

Path to competition files: /kaggle/input/competitions/smart-mcq-solver-challenge


In [4]:

import numpy as np
import pandas as pd
import torch
from dataclasses import dataclass
from typing import Optional, Union
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer,
)
from transformers.tokenization_utils_base import PreTrainedTokenizerBase
from peft import LoraConfig, get_peft_model, TaskType
from transformers import set_seed
from sklearn.model_selection import StratifiedKFold


# -----------------------------
# Config
# -----------------------------
MODEL_NAME = "microsoft/deberta-v3-large"   # bigger backbone, LoRA keeps it trainable
TRAIN_CSV = "/kaggle/input/datasets/chinmayeemilindawale/train-csv/train.csv"
TEST_CSV = "/kaggle/input/datasets/chinmayeemilindawale/csv-test/test.csv"
OUTPUT_DIR = "/kaggle/working/roberta_deberta"
MAX_LEN = 256
OPTIONS = ["A", "B", "C", "D", "E"]
LABEL2ID = {c: i for i, c in enumerate(OPTIONS)}
ID2LABEL = {i: c for c, i in LABEL2ID.items()}

# target_modules differ by architecture family:
#   DeBERTa-v3 -> "query_proj", "value_proj"
#   BERT/RoBERTa -> "query", "value"
LORA_TARGET_MODULES = ["query_proj", "value_proj"] if "deberta" in MODEL_NAME else ["query", "value"]

USE_WANDB = False
if USE_WANDB:
    import wandb
    wandb.init(project="mcq-science-exam", name="deberta-v3-large-lora")
    

In [5]:

# -----------------------------
# Load & prep data
# -----------------------------
def load_data():
    train_df = pd.read_csv(TRAIN_CSV)
    test_df = pd.read_csv(TEST_CSV)
    train_df["label"] = train_df["answer"].map(LABEL2ID)
    return train_df, test_df


def to_hf_dataset(df, has_label=True):
    cols = ["id", "prompt"] + OPTIONS + (["label"] if has_label else [])
    return Dataset.from_pandas(df[cols].reset_index(drop=True))


def preprocess(examples, tokenizer):
    first_sentences = [[p] * 5 for p in examples["prompt"]]
    second_sentences = [
        [examples[opt][i] for opt in OPTIONS] for i in range(len(examples["prompt"]))
    ]
    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])

    tokenized = tokenizer(
        first_sentences,
        second_sentences,
        truncation=True,
        max_length=MAX_LEN,
        padding=False,
    )
    return {k: [v[i : i + 5] for i in range(0, len(v), 5)] for k, v in tokenized.items()}


@dataclass
class DataCollatorForMultipleChoice:
    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str] = True
    max_length: Optional[int] = None

    def __call__(self, features):
        label_name = "label" if "label" in features[0].keys() else None
        labels = [feature.pop(label_name) for feature in features] if label_name else None
        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])
        flattened_features = [
            [{k: v[i] for k, v in feature.items()} for i in range(num_choices)]
            for feature in features
        ]
        flattened_features = sum(flattened_features, [])

        batch = self.tokenizer.pad(
            flattened_features,
            padding=self.padding,
            max_length=self.max_length,
            return_tensors="pt",
        )
        batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
        if labels is not None:
            batch["labels"] = torch.tensor(labels, dtype=torch.int64)
        return batch


def map_at_3(logits, labels):
    top3 = np.argsort(-logits, axis=1)[:, :3]
    scores = []
    for pred_row, true_label in zip(top3, labels):
        if true_label == pred_row[0]:
            scores.append(1.0)
        elif true_label == pred_row[1]:
            scores.append(0.5)
        elif true_label == pred_row[2]:
            scores.append(1.0 / 3)
        else:
            scores.append(0.0)
    return np.mean(scores)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = (preds == labels).mean()
    return {"accuracy": acc, "map@3": map_at_3(logits, labels)}


In [6]:

def main():
    train_df, test_df = load_data()
    tr_df, val_df = train_test_split(
    train_df,
    test_size=0.15,
    random_state=42,
    stratify=train_df["label"]
)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    set_seed(42)
    base_model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)

    # ---- Wrap with LoRA ----
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,   # closest available task type; MC head behaves like a scoring head
        r=16,
        lora_alpha=132,
        lora_dropout=0.1,
        target_modules=LORA_TARGET_MODULES,
        modules_to_save=["classifier", "pooler"],
        bias="none",
    )
    model = get_peft_model(base_model, lora_config)
    model.print_trainable_parameters()# sanity check: should show a tiny % of total params


    train_ds = to_hf_dataset(tr_df).map(
        lambda x: preprocess(x, tokenizer), batched=True, remove_columns=["prompt"] + OPTIONS + ["id"]
    )
    val_ds = to_hf_dataset(val_df).map(
        lambda x: preprocess(x, tokenizer), batched=True, remove_columns=["prompt"] + OPTIONS + ["id"]
    )
    test_ds = to_hf_dataset(test_df, has_label=False).map(
        lambda x: preprocess(x, tokenizer), batched=True, remove_columns=["prompt"] + OPTIONS + ["id"]
    )
    collator = DataCollatorForMultipleChoice(tokenizer=tokenizer)

    args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="map@3",
        learning_rate=5e-5,              # LoRA typically wants a higher LR than full fine-tune
        per_device_train_batch_size=2,   # smaller since backbone is larger
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=8,
        num_train_epochs=8,              # LoRA often needs a few more epochs to converge
        weight_decay=0.05,
        warmup_ratio=0.1,
        logging_steps=20,
        save_total_limit=1,
        report_to=["wandb"] if USE_WANDB else [],
        fp16=False,
        bf16=False,                      # keep mixed precision off, same instability risk as full fine-tune
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer,
        data_collator=collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    print("Validation results:", trainer.evaluate())

    preds = trainer.predict(test_ds)
    logits = preds.predictions
    top3_idx = np.argsort(-logits, axis=1)[:, :3]
    top3_letters = [" ".join(ID2LABEL[i] for i in row) for row in top3_idx]

    submission = pd.DataFrame({"id": test_df["id"], "prediction": top3_letters})
    submission.to_csv("/kaggle/working/roberta_deberta/submission.csv", index=False)
    print(submission.head())
    print("Saved submission to /kaggle/working/roberta_deberta/submission.csv")

    # save just the LoRA adapter (small, a few MB) rather than the full backbone
    model.save_pretrained("./mcq_lora_adapter")

    if USE_WANDB:
        wandb.finish()


if __name__ == "__main__":
    main()



config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/874M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/874M [00:00<?, ?B/s]

...

Epoch,Training Loss,Validation Loss,Accuracy,Map@3
1,26.849902,3.058594,0.510000,0.662222
2,23.954248,2.181641,0.703333,0.811667
3,11.440717,0.738770,0.896667,0.934444
4,7.926956,0.362549,0.956667,0.970000
5,5.603197,0.250488,0.973333,0.985000
6,4.252784,0.191528,0.986667,0.993333
7,3.166961,0.146973,0.996667,0.998333
8,3.465977,0.154907,0.996667,0.998333


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector

Training Loss,Validation Loss,Epoch,Accuracy,Map@3
3.465977,0.146973,8,0.996667,0.998333


Validation results: {'eval_loss': 0.14697265625, 'eval_accuracy': 0.9966666666666667, 'eval_map@3': 0.9983333333333333}


   id prediction
0   1      A E D
1   2      B C A
2   3      B E D
3   4      E A C
4   5      C D A
Saved submission to /kaggle/working/roberta_deberta/submission.csv
